# GET SYSTEM/SCREEN INFO

In [1]:
import ctypes
import platform

def get_screen_dpi():
    """Gets screen's DPI depending on the OS."""
    if platform.system() == "Windows":
        # Para Windows
        hdc = ctypes.windll.user32.GetDC(0)
        dpi = ctypes.windll.gdi32.GetDeviceCaps(hdc, 88)  # 88 = LOGPIXELSX
        ctypes.windll.user32.ReleaseDC(0, hdc)
        return dpi
    # elif platform.system() == "Darwin":
    #     # Para macOS (requiere Quartz)
    #     try:
    #         from AppKit import NSScreen
    #         screen = NSScreen.mainScreen()
    #         return screen.backingScaleFactor() * 72  # 72 es el DPI base en macOS
    #     except ImportError:
    #         return "No se pudo obtener los DPI en macOS. Instala pyobjc-framework-AppKit."
    # elif platform.system() == "Linux":
    #     # Para Linux, se obtiene desde xrandr
    #     try:
    #         import subprocess
    #         output = subprocess.check_output("xrdb -query", shell=True).decode()
    #         for line in output.split("\n"):
    #             if "Xft.dpi" in line:
    #                 return float(line.split(":")[-1].strip())
    #     except Exception:
    #         return "No se pudo obtener los DPI en Linux."
    else:
        return "OS not supported."

# Ejemplo de uso
print(f"{platform.system()} - Screen DPI: {get_screen_dpi()}")


Windows - Screen DPI: 96


# GENERATE IMAGES

In [1]:
import os
import numpy as np
from skimage import io, img_as_ubyte


In [2]:

def cpd_to_radius(cpd, image_width, screen_width_cm, distance_to_screen_cm, dpi):
    """Convert CPD to radius for the Fourier transform."""
    pixels_per_cm = dpi / 2.54  # DPI to px/cm
    pixels_per_degree = (distance_to_screen_cm * np.tan(np.radians(1))) * pixels_per_cm
    cycles_per_pixel = cpd / pixels_per_degree
    return int(cycles_per_pixel * (image_width / 2))

def normalize_image(image):
    """Normalizes the image to a range of 0-255 (uint8)."""
    image_normalized = np.real(image)
    image_normalized -= image_normalized.min()
    image_normalized /= image_normalized.max()
    return img_as_ubyte(image_normalized)

def bandpass_filter_image(image, mask, min_cpd, max_cpd, image_width, screen_width_cm, distance_to_screen_cm, dpi):
    """Aplica un filtro paso banda entre min_cpd y max_cpd."""
    min_radius = cpd_to_radius(min_cpd, image_width, screen_width_cm, distance_to_screen_cm, dpi)
    max_radius = cpd_to_radius(max_cpd, image_width, screen_width_cm, distance_to_screen_cm, dpi)

    # Fourier Transform
    f_transform = np.fft.fft2(image * mask)
    f_transform_shifted = np.fft.fftshift(f_transform)

    # Crear máscara paso banda
    rows, cols = image.shape
    crow, ccol = rows // 2, cols // 2
    y, x = np.ogrid[:rows, :cols]
    distance_squared = (x - ccol) ** 2 + (y - crow) ** 2

    bandpass_mask = np.logical_and(distance_squared >= min_radius**2, distance_squared <= max_radius**2).astype(float)

    # Aplicar filtro
    bandpass_frequencies = f_transform_shifted * bandpass_mask

    # Transformada inversa
    filtered_image = np.fft.ifft2(np.fft.ifftshift(bandpass_frequencies))

    return normalize_image(filtered_image)

def process_image(image, mask, cpd_values, output_dir, image_file, image_width):
    """Apply filters and save images."""
    for cpd in cpd_values:
        radius = cpd_to_radius(cpd, image_width, screen_width_cm, distance_to_screen_cm, screen_resolution_dpi)

        low_output_dir = os.path.join(output_dir, f"low_cpd_{cpd}")
        high_output_dir = os.path.join(output_dir, f"high_cpd_{cpd}")
        os.makedirs(low_output_dir, exist_ok=True)
        os.makedirs(high_output_dir, exist_ok=True)

        # Fourier Transform
        f_transform = np.fft.fft2(image * mask)
        f_transform_shifted = np.fft.fftshift(f_transform)

        # Low-pass and high-pass filters
        rows, cols = image.shape
        crow, ccol = rows // 2, cols // 2
        y, x = np.ogrid[:rows, :cols]
        low_pass_filter = np.zeros((rows, cols))
        low_pass_filter[(x - ccol) ** 2 + (y - crow) ** 2 <= radius ** 2] = 1
        high_pass_filter = 1 - low_pass_filter

        # Apply filters
        low_frequencies = f_transform_shifted * low_pass_filter
        high_frequencies = f_transform_shifted * high_pass_filter

        # Inverse Fourier Transform
        low_image = np.fft.ifft2(np.fft.ifftshift(low_frequencies))
        high_image = np.fft.ifft2(np.fft.ifftshift(high_frequencies))

        # Save images
        io.imsave(os.path.join(low_output_dir, f"low_{image_file}"), normalize_image(low_image))
        io.imsave(os.path.join(high_output_dir, f"high_{image_file}"), normalize_image(high_image))

        print(f"Processed {image_file} with CPD {cpd} (radius {radius})")

def process_image_bandpass(image, mask, cpd_ranges, output_dir, image_file, image_width):
    """Aplica filtros paso banda con rangos de CPD y guarda imágenes."""
    for (min_cpd, max_cpd) in cpd_ranges:
        band_output_dir = os.path.join(output_dir, f"bandpass_{min_cpd}_{max_cpd}_cpd")
        os.makedirs(band_output_dir, exist_ok=True)

        filtered_image = bandpass_filter_image(
            image, mask, min_cpd, max_cpd, image_width,
            screen_width_cm, distance_to_screen_cm, screen_resolution_dpi
        )

        io.imsave(os.path.join(band_output_dir, f"band_{image_file}"), filtered_image)
        print(f"Processed {image_file} with CPD range {min_cpd}-{max_cpd}")

def extract_spatial_frequencies(input_dir, cpd_values):
    """Scrolls through the directory and extracts the spatial frequencies of all images."""
    for root, _, files in os.walk(input_dir):
        output_dir = os.path.join(root, "Processed_SF")
        os.makedirs(output_dir, exist_ok=True)

        for image_file in files:
            if image_file.endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                image_path = os.path.join(root, image_file)
                image = io.imread(image_path, as_gray=True)
                mask = np.ones_like(image)

                process_image(image, mask, cpd_values, output_dir, image_file, image.shape[1])
                
def extract_spatial_frequencies_bandpass(input_dir, cpd_ranges):
    for root, _, files in os.walk(input_dir):
        output_dir = os.path.join(root, "Processed_Bandpass_SF")
        os.makedirs(output_dir, exist_ok=True)

        for image_file in files:
            if image_file.endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                image_path = os.path.join(root, image_file)
                image = io.imread(image_path, as_gray=True)
                mask = np.ones_like(image)

                process_image_bandpass(image, mask, cpd_ranges, output_dir, image_file, image.shape[1])



In [3]:
# CONFIG:

cpd_values = [5,6,7,8] 
distance_to_screen_cm = 57  
screen_resolution_dpi = 244 # Get the value from: https://pixelcalculator.com/es OR the function above (maybe not working...)
screen_width_cm = 30

# IMAGES PATH
input_base_dir = r"C:\Users\akoun\Desktop\Biocruces\2.DATASETS\SiBurmuin_50_images_dataset\processed\animals\raw"

In [ ]:
# BANDPASS FILTERS EXAMPLE:
# Parámetros de visualización

screen_width_cm = 40         # Ancho físico de la pantalla en cm
distance_to_screen_cm = 57   # Distancia desde el observador a la pantalla en cm
screen_resolution_dpi = 244   # Resolución de la pantalla en DPI # Get the value from: https://pixelcalculator.com/es

cpd_ranges = [(1, 3), (4, 6), (10, 18)]  # Ejemplo de filtros paso banda

extract_spatial_frequencies_bandpass(input_base_dir, cpd_ranges)

Processed B_N209044.jpg with CPD range 1-3
Processed B_N209044.jpg with CPD range 4-6
Processed B_N209044.jpg with CPD range 10-18
Processed B_N253041.jpg with CPD range 1-3
Processed B_N253041.jpg with CPD range 4-6
Processed B_N253041.jpg with CPD range 10-18
Processed B_N253081.jpg with CPD range 1-3
Processed B_N253081.jpg with CPD range 4-6
Processed B_N253081.jpg with CPD range 10-18
Processed B_N253099.jpg with CPD range 1-3
Processed B_N253099.jpg with CPD range 4-6
Processed B_N253099.jpg with CPD range 10-18
Processed B_N289048.jpg with CPD range 1-3
Processed B_N289048.jpg with CPD range 4-6
Processed B_N289048.jpg with CPD range 10-18
Processed B_N427024.jpg with CPD range 1-3
Processed B_N427024.jpg with CPD range 4-6
Processed B_N427024.jpg with CPD range 10-18
Processed B_N771063.jpg with CPD range 1-3
Processed B_N771063.jpg with CPD range 4-6
Processed B_N771063.jpg with CPD range 10-18
Processed B_N815060.jpg with CPD range 1-3
Processed B_N815060.jpg with CPD range 4

In [6]:
# RUN
extract_spatial_frequencies(input_base_dir, cpd_values)


Processed B_N209044.jpg with CPD 5 (radius 6)
Processed B_N209044.jpg with CPD 6 (radius 8)
Processed B_N209044.jpg with CPD 7 (radius 9)
Processed B_N209044.jpg with CPD 8 (radius 10)
Processed B_N253041.jpg with CPD 5 (radius 6)
Processed B_N253041.jpg with CPD 6 (radius 8)
Processed B_N253041.jpg with CPD 7 (radius 9)
Processed B_N253041.jpg with CPD 8 (radius 10)
Processed B_N253081.jpg with CPD 5 (radius 6)
Processed B_N253081.jpg with CPD 6 (radius 8)
Processed B_N253081.jpg with CPD 7 (radius 9)
Processed B_N253081.jpg with CPD 8 (radius 10)
Processed B_N253099.jpg with CPD 5 (radius 6)
Processed B_N253099.jpg with CPD 6 (radius 8)
Processed B_N253099.jpg with CPD 7 (radius 9)
Processed B_N253099.jpg with CPD 8 (radius 10)
Processed B_N289048.jpg with CPD 5 (radius 6)
Processed B_N289048.jpg with CPD 6 (radius 8)
Processed B_N289048.jpg with CPD 7 (radius 9)
Processed B_N289048.jpg with CPD 8 (radius 10)
Processed B_N427024.jpg with CPD 5 (radius 6)
Processed B_N427024.jpg with 

# GENERATE MOSAIC (not bmp files but can be easily adapted)

In [8]:
from PIL import Image
import os
import math

def add_white_background(img): # if RGBA, add white background
    if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
        background = Image.new('RGB', img.size, background_color)
        background.paste(img, mask=img.split()[3])
        return background
    else:
        return img.convert('RGB')

def create_mosaic(folder_path, output_path):
    image_files = [os.path.join(folder_path, file) for file in os.listdir(folder_path) if file.endswith(('png', 'jpg', 'jpeg','bmp'))]
    
    if not image_files:
        return
    
    grid_size = math.ceil(math.sqrt(len(image_files)))
    mosaic_width = grid_size * tile_size[0] + (grid_size + 1) * padding
    mosaic_height = grid_size * tile_size[1] + (grid_size + 1) * padding
    
    
    mosaic_image = Image.new('RGB', (mosaic_width, mosaic_height), background_color)
    
    # PASTE IMAGES IN GRID
    for idx, image_file in enumerate(image_files):
        row = idx // grid_size
        col = idx % grid_size
        img = Image.open(image_file)
        img = add_white_background(img)
        img.thumbnail(tile_size)  # KEEP ASPECT RATIO
        
        x_offset = col * tile_size[0] + (col + 1) * padding
        y_offset = row * tile_size[1] + (row + 1) * padding
        mosaic_image.paste(img, (x_offset, y_offset))
    
    
    mosaic_image.save(output_path)
    print(f"Mosaic saved in: {output_path}")



In [9]:

# PROCESSED IMAGES DATASET PATH
dataset_dir = r"C:\Users\akoun\Desktop\Biocruces\2.DATASETS\SiBurmuin_50_images_dataset\processed\animals\raw\Processed_Bandpass_SF"

# MOSAIC PARAMS
tile_size = (100, 100)  # IMG SIZE --> WARNING: CPD WILL CHANGE
padding = 10 
background_color = (255, 255, 255)


# Iterate over folders and generate mosaic
for folder in os.listdir(dataset_dir):
    folder_path = os.path.join(dataset_dir, folder)
    if os.path.isdir(folder_path):
        output_image = os.path.join(dataset_dir, f"{folder}_mosaic.jpg")
        create_mosaic(folder_path, output_image)


Mosaic saved in: C:\Users\akoun\Desktop\Biocruces\2.DATASETS\SiBurmuin_50_images_dataset\processed\animals\raw\Processed_Bandpass_SF\bandpass_10_18_cpd_mosaic.jpg
Mosaic saved in: C:\Users\akoun\Desktop\Biocruces\2.DATASETS\SiBurmuin_50_images_dataset\processed\animals\raw\Processed_Bandpass_SF\bandpass_1_3_cpd_mosaic.jpg
Mosaic saved in: C:\Users\akoun\Desktop\Biocruces\2.DATASETS\SiBurmuin_50_images_dataset\processed\animals\raw\Processed_Bandpass_SF\bandpass_4_6_cpd_mosaic.jpg


# VERIFY CORRECT SIZE

In [ ]:
import tkinter as tk
from PIL import Image

# def get_screen_dpi():
#     """Obtiene los DPI de la pantalla"""
#     root = tk.Tk()
#     dpi_x = root.winfo_fpixels('1i')  # DPI horizontal
#     dpi_y = root.winfo_fpixels('1i')  # DPI vertical
#     root.destroy()
#     return dpi_x, dpi_y

def get_image_size_cm(image_path):
    """calculates the size in centimeters that an image should occupy on the screen"""
    dpi_x, dpi_y = 244,244#get_screen_dpi()  #GET SCREEN DPI --> https://pixelcalculator.com/es OR the function above (maybe not working...)
    
    with Image.open(image_path) as img:
        width_px, height_px = img.size  # GET SIZE IN PX
    
    width_cm = (width_px / dpi_x) * 2.54  # PX to CM
    height_cm = (height_px / dpi_y) * 2.54
    
    print(f"The image {image_path} should measure aprox:")
    print(f"- Width: {width_cm:.2f} cm")
    print(f"- Height: {height_cm:.2f} cm")
    return width_cm, height_cm


path = r"PATH   TO  IMAGE"
get_image_size_cm(path)
